Session 4 — Cleaning and Feature Engineering

Input: the two raw files from Sessions 2 and 3.
Output: one clean artist level table, ready for the charts

What this notebook does
1. It loads the raw discography (API) and artist bios (scraping) files.
2. Prints a data quality report for each.
3. Cleans types, dates, and text fields with a reason for every decision.
4. Engineers the career bump features: releases-per-year before vs after Eurovision.
5. Merges everything into one artist-level table and saves it to data/processed/


Uploading the two raw CSV files

In [2]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving eurovision_artists_discography_20260520_092643 (1).csv to eurovision_artists_discography_20260520_092643 (1).csv
Saving eurovision_artist_bios_20260520_093136 (2).csv to eurovision_artist_bios_20260520_093136 (2) (1).csv
Uploaded: ['eurovision_artists_discography_20260520_092643 (1).csv', 'eurovision_artist_bios_20260520_093136 (2) (1).csv']


Load both files into DataFrames

In [3]:
import pandas as pd
import numpy as np
import re
from datetime import datetime


disco_name = [n for n in uploaded if "discography" in n][0]
bios_name  = [n for n in uploaded if "bios" in n][0]

disco = pd.read_csv(disco_name)
bios  = pd.read_csv(bios_name)

print("Discography:", disco.shape)
print("Bios:", bios.shape)

Discography: (2793, 11)
Bios: (90, 12)


Data quality report, before changing anything, we look at what we have: shapes, types, missing values, duplicates, and a numeric summary.

In [4]:
print("=== DISCOGRAPHY ===")
print("Shape:", disco.shape)
print()
disco.info()
print()
print("Missing values per column:")
print(disco.isna().sum())
print()
print("Duplicate rows:", disco.duplicated().sum())

=== DISCOGRAPHY ===
Shape: (2793, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2793 entries, 0 to 2792
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   artist              2793 non-null   object
 1   eurovision_year     2793 non-null   int64 
 2   eurovision_song     2793 non-null   object
 3   eurovision_country  2793 non-null   object
 4   eurovision_rank     2793 non-null   int64 
 5   artist_mbid         2793 non-null   object
 6   release_id          2793 non-null   object
 7   release_title       2793 non-null   object
 8   release_date        2718 non-null   object
 9   release_country     2367 non-null   object
 10  release_status      2738 non-null   object
dtypes: int64(2), object(9)
memory usage: 240.2+ KB

Missing values per column:
artist                  0
eurovision_year         0
eurovision_song         0
eurovision_country      0
eurovision_rank         0
artist_mbid         

In [5]:
print("=== ARTIST BIOS ===")
print("Shape:", bios.shape)
print()
bios.info()
print()
print("Missing values per column:")
print(bios.isna().sum())
print()
print("Duplicate rows:", bios.duplicated().sum())

=== ARTIST BIOS ===
Shape: (90, 12)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   artist                    90 non-null     object
 1   earliest_eurovision_year  90 non-null     int64 
 2   earliest_eurovision_rank  90 non-null     int64 
 3   wikipedia_slug            90 non-null     object
 4   wikipedia_url             90 non-null     object
 5   scraped_at                90 non-null     object
 6   born                      69 non-null     object
 7   origin                    42 non-null     object
 8   genres                    83 non-null     object
 9   occupation                66 non-null     object
 10  years_active              81 non-null     object
 11  labels                    58 non-null     object
dtypes: int64(2), object(10)
memory usage: 8.6+ KB

Missing values per column:
artist                   

Clean the discography (releases) table

Decision 1 — extract a release year. release_date comes in mixed forms (2015, 2015-03, 2015-03-12) and some are blank. We take the first 4 characters as the year, which works for all three forms. We drop releases with no usable year, because a release we can't place in time is useless for a before/after analysis.

Decision 2 — keep only 'Official' releases. MusicBrainz lists bootlegs and promos too; counting those would inflate an artist's output. We keep release_status == "Official" (and rows where status is missing, to avoid losing real releases).

Decision 3 — remove duplicate releases. Artists who placed top-10 in two different years (e.g. Sergey Lazarev) appear twice in the raw data with the same releases. We de-duplicate on artist + release_id.

In [6]:

disco["release_year"] = pd.to_numeric(
    disco["release_date"].astype(str).str[:4], errors="coerce"
)
before_drop = len(disco)
disco = disco[disco["release_year"].notna()].copy()
disco["release_year"] = disco["release_year"].astype(int)
print(f"Dropped {before_drop - len(disco)} releases with no usable year.")


disco = disco[
    disco["release_status"].isna() | (disco["release_status"] == "Official")
].copy()
print("Rows after keeping Official:", len(disco))


disco = disco.drop_duplicates(subset=["artist", "release_id"]).copy()
print("Rows after de-duplicating releases:", len(disco))

Dropped 75 releases with no usable year.
Rows after keeping Official: 2637
Rows after de-duplicating releases: 2637



Clean the artist-bios table

Decision 4 — career_start_year from years_active. This field is free text like 2004–present or 2012–2016, 2019–present. We pull out the first 4-digit year as the career start.

Decision 5 — birth_year from born. The born field mixes a date and a place. We extract the first 4-digit year as the birth year and leave the rest.

Decision 6 — primary_genre from genres. genres is a comma-separated string. We take the first genre as the primary one and count how many genres are listed. Missing genres become "Unknown".

In [7]:
def first_year(text):
    """Return the first 4-digit year found in a string, or NaN."""
    if pd.isna(text):
        return np.nan
    match = re.search(r"(19|20)\d{2}", str(text))
    return int(match.group()) if match else np.nan


bios["career_start_year"] = bios["years_active"].apply(first_year)
bios["birth_year"]        = bios["born"].apply(first_year)


bios["genres"] = bios["genres"].fillna("Unknown")
bios["primary_genre"] = bios["genres"].str.split(",").str[0].str.strip().str.lower()
bios["genre_count"]   = bios["genres"].apply(
    lambda g: 0 if g == "Unknown" else len(str(g).split(","))
)

print(bios[["artist", "years_active", "career_start_year", "birth_year", "primary_genre"]].head(10))

                      artist  years_active  career_start_year  birth_year  \
0             Måns Zelmerlöw  2005–present             2005.0      1986.0   
1            Polina Gagarina  2003–present             2003.0      1987.0   
2                    Il Volo  2010–present             2010.0         NaN   
3                Loïc Nottet  2014–present             2014.0      1996.0   
4           Aminata Savadogo  2008–present             2008.0      1993.0   
5              Guy Sebastian           NaN                NaN      1981.0   
6  Elina Born and Stig Rästa  2012–present             2012.0      1994.0   
7            Bojana Stamenov  2009-present             2009.0      1986.0   
8                Nadav Guedj  2015-present             2015.0      1998.0   
9                     Jamala  2001–present             2001.0      1983.0   

                                    primary_genre  
0                                             pop  
1                                             po

Engineer the career-bump features

For every artist we need: how many releases per year before their Eurovision appearance, and how many per year after.

Decision 7 — one Eurovision event per artist. A few artists placed twice; we use their earliest top-10 year as the event we measure around.

Decision 8 — before / after counts. Releases strictly before the event year count as "before"; strictly after count as "after". The event year itself is excluded (it's the boost moment, not a clean before or after).

Decision 9 — rates, not raw counts. A 2015 artist has had 10 years to release more music; a 2024 artist has had 1. So we divide by the number of years available on each side to get a fair releases-per-year rate.

In [8]:
CURRENT_YEAR = datetime.now().year


event_year = disco.groupby("artist")["eurovision_year"].min().rename("event_year")
event_rank = disco.groupby("artist")["eurovision_rank"].min().rename("event_rank")
event_country = disco.groupby("artist")["eurovision_country"].first().rename("event_country")


disco = disco.merge(event_year, on="artist", how="left")


disco["is_before"] = disco["release_year"] < disco["event_year"]
disco["is_after"]  = disco["release_year"] > disco["event_year"]

agg = disco.groupby("artist").agg(
    total_releases=("release_id", "count"),
    releases_before=("is_before", "sum"),
    releases_after=("is_after", "sum"),
    first_release_year=("release_year", "min"),
).reset_index()


agg = agg.merge(event_year, on="artist").merge(event_rank, on="artist").merge(event_country, on="artist")
print(agg.head())

            artist  total_releases  releases_before  releases_after  \
0       Alessandra              58               44              14   
1            Alika              29               29               0   
2  Amanda Tenfjord              25               17               0   
3             Amir              38                1              31   
4   Angelina Mango              49               18               7   

   first_release_year  event_year  event_rank event_country  
0                2009        2023           5        Norway  
1                2000        2023           8       Estonia  
2                2014        2022           8        Greece  
3                2013        2016           6        France  
4                2020        2024           7         Italy  


## 6. Merge with bios and compute the rates

In [9]:

artist = agg.merge(
    bios[["artist", "career_start_year", "birth_year", "primary_genre", "genre_count", "origin"]],
    on="artist", how="left",
)


artist["career_start_year"] = artist["career_start_year"].fillna(artist["first_release_year"])


artist["years_before"] = artist["event_year"] - artist["career_start_year"]
artist["years_after"]  = CURRENT_YEAR - artist["event_year"]


artist["years_before"] = artist["years_before"].where(artist["years_before"] > 0, np.nan)
artist["years_after"]  = artist["years_after"].where(artist["years_after"] > 0, np.nan)


artist["releases_per_year_before"] = artist["releases_before"] / artist["years_before"]
artist["releases_per_year_after"]  = artist["releases_after"]  / artist["years_after"]


artist["bump_ratio"] = (
    artist["releases_per_year_after"] / artist["releases_per_year_before"]
)


def rank_group(r):
    if r == 1:
        return "Winner"
    elif r <= 3:
        return "Runner-up"
    else:
        return "Top-10"
artist["rank_group"] = artist["event_rank"].apply(rank_group)


artist["enough_after_data"] = artist["years_after"] >= 2

print(artist.shape)
artist.head()

(83, 20)


,artist,total_releases,releases_before,releases_after,first_release_year,event_year,event_rank,event_country,career_start_year,birth_year,primary_genre,genre_count,origin,years_before,years_after,releases_per_year_before,releases_per_year_after,bump_ratio,rank_group,enough_after_data
0,Alessandra,58,44,14,2009,2023,5,Norway,2022.0,2002.0,edm pop,1.0,"Lillehammer , Norway",1.0,3,44.000000,4.666667,0.106061,Top-10,True
1,Alika,29,29,0,2000,2023,8,Estonia,2021.0,2002.0,unknown,0.0,NaN,2.0,3,14.500000,0.000000,0.000000,Top-10,True
2,Amanda Tenfjord,25,17,0,2014,2022,8,Greece,2014.0,1997.0,synth-pop [ 2 ] indie pop [ 3 ] nordic pop [ 3 ],1.0,"Tennfjord , Norway",8.0,4,2.125000,0.000000,0.000000,Top-10,True
3,Amir,38,1,31,2013,2016,6,France,2013.0,NaN,NaN,NaN,NaN,3.0,10,0.333333,3.100000,9.300000,Top-10,True
4,Angelina Mango,49,18,7,2020,2024,7,Italy,2020.0,2001.0,pop,1.0,NaN,4.0,2,4.500000,3.500000,0.777778,Top-10,True


Final missing-value pass, we make the missing-value strategy explicit, column by column.

In [10]:

artist["primary_genre"] = artist["primary_genre"].fillna("Unknown")


print("Artists missing birth_year:", artist["birth_year"].isna().sum())


print("Artists with no computable bump_ratio:", artist["bump_ratio"].isna().sum())
print("  (these are mostly newcomers who debuted at Eurovision — interesting in themselves)")


print()
print(artist.isna().sum())

Artists missing birth_year: 29
Artists with no computable bump_ratio: 8
  (these are mostly newcomers who debuted at Eurovision — interesting in themselves)

artist                       0
total_releases               0
releases_before              0
releases_after               0
first_release_year           0
event_year                   0
event_rank                   0
event_country                0
career_start_year            0
birth_year                  29
primary_genre                0
genre_count                  9
origin                      47
years_before                 4
years_after                  0
releases_per_year_before     4
releases_per_year_after      0
bump_ratio                   8
rank_group                   0
enough_after_data            0
dtype: int64


Save the clean dataset

The artist-level table is the main analytical dataset after. We also save a tidy release-level table (with the cleaned release_year) for the time-series chart.

In [11]:
import os
os.makedirs("processed", exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

clean_artist_csv = f"processed/clean_artists_{stamp}.csv"
clean_releases_csv = f"processed/clean_releases_{stamp}.csv"

artist.to_csv(clean_artist_csv, index=False)

disco[["artist", "event_year", "release_year", "release_title", "release_country"]].to_csv(
    clean_releases_csv, index=False
)

print("Saved:", clean_artist_csv, artist.shape)
print("Saved:", clean_releases_csv, disco.shape)

Saved: processed/clean_artists_20260522_083307.csv (83, 20)
Saved: processed/clean_releases_20260522_083307.csv (2637, 15)


Cleaning summary, a short before/after of the whole process.

In [12]:
print("CLEANING SUMMARY")
print("-" * 40)
print(f"Raw discography rows:       {before_drop}")
print(f"Clean release rows:         {len(disco)}")
print(f"Raw artist-bio rows:        {len(bios)}")
print(f"Final artist-level rows:    {len(artist)}")
print()
print("Columns engineered (new):")
for c in ["release_year", "career_start_year", "birth_year", "primary_genre",
          "genre_count", "years_before", "years_after",
          "releases_per_year_before", "releases_per_year_after",
          "bump_ratio", "rank_group", "enough_after_data"]:
    print("  +", c)
print()
print("Artists per rank group:")
print(artist["rank_group"].value_counts())

CLEANING SUMMARY
----------------------------------------
Raw discography rows:       2793
Clean release rows:         2637
Raw artist-bio rows:        90
Final artist-level rows:    83

Columns engineered (new):
  + release_year
  + career_start_year
  + birth_year
  + primary_genre
  + genre_count
  + years_before
  + years_after
  + releases_per_year_before
  + releases_per_year_after
  + bump_ratio
  + rank_group
  + enough_after_data

Artists per rank group:
rank_group
Top-10       60
Runner-up    13
Winner       10
Name: count, dtype: int64


Download the clean files

In [14]:
from google.colab import files
files.download(clean_artist_csv)
files.download(clean_releases_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>